<a href="https://colab.research.google.com/github/NguyenBaTam-tristan/deepml-curriculum-scraper/blob/main/deep_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cài đặt Playwright, browser và TẤT CẢ thư viện hệ thống cần thiết
!pip install playwright nest_asyncio
!playwright install chromium
!playwright install-deps chromium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 623.8/623.8 kB 48.4 MB/s eta 0:00:00
186.8 MiB [] 0% 379.5s186.8 MiB [] 0% 27.9s186.8 MiB [] 0% 16.4s186.8 MiB [] 0% 14.2s186.8 MiB [] 0% 7.8s186.8 MiB [] 1% 4.6s186.8 MiB [] 2% 3.6s186.8 MiB [] 3% 3.5s186.8 MiB [] 3% 3.3s186.8 MiB [] 4% 3.2s186.8 MiB [] 5% 2.9s186.8 MiB [] 6% 2.7s186.8 MiB [] 6% 2.9s186.8 MiB [] 7% 3.0s186.8 MiB [] 7% 3.1s186.8 MiB [] 8% 3.0s186.8 MiB [] 9% 2.8s186.8 MiB [] 10% 2.6s186.8 MiB [] 11% 2.5s186.8 MiB [] 12% 2.4s186.8 MiB [] 13% 2.3s186.8 MiB [] 14% 2.3s186.8 MiB [] 15% 2.2s186.8 MiB [] 16% 2.1s186.8 MiB [] 17% 2.1s186.8 MiB [] 18% 2.0s186.8 MiB [] 19% 2.0s186.8 MiB [] 19% 2.1s186.8 MiB [] 19% 2.2s186.8 MiB [] 20% 2.2s186.8 MiB [] 21% 2.1s186.8 MiB [] 22% 2.0s186.8 MiB [] 23% 2.0s186.8 MiB [] 24% 1.9s186.8 MiB [] 25% 1.9s186.8 MiB [] 25% 2.0s186.8 MiB [] 27% 1.9s186.8 MiB [] 28% 1.8s186.8 MiB [] 29% 1.7s186.8 MiB [] 31% 1.7s186.8 MiB []

In [ ]:
"""
Deep-ML scraper v2 (dựa trên cấu trúc DOM thật từ ảnh DevTools).

COLAB:
    !pip -q install playwright nest_asyncio
    !playwright install chromium
    !playwright install-deps chromium
  rồi dán toàn bộ file này vào 1 cell và chạy (dòng cuối gọi run()).
  Nếu run() báo lỗi event loop, thay dòng cuối bằng:  await main()

MÁY LOCAL:
    pip install playwright && playwright install chromium
    python deepml_scraper_v2.py
"""
import asyncio
import csv
import json
import re
import time

from playwright.async_api import async_playwright

BASE = "https://www.deep-ml.com"
MAX_PAGES = 400

SECTIONS = {
    "problems": {"url": BASE + "/problems", "mode": "table"},
    "math":     {"url": BASE + "/math-problems", "mode": "cards", "prefix": BASE + "/math-problems"},
    "labs":     {"url": BASE + "/labs", "mode": "cards", "prefix": BASE + "/labs"},
    "projects": {"url": BASE + "/projects", "mode": "cards", "prefix": BASE + "/projects"},
}

DIFF_RE = re.compile(r"\b(easy|medium|hard|beginner|intermediate|advanced)\b", re.I)
NUM_RE = re.compile(r"^[\d\s.,%/+\-:]+$")

# ------------------------------------------------------------------ JS
JS_TABLE = """
() => [...document.querySelectorAll('tr[data-testid^="problem-"]')].map(tr => {
  const tid = tr.getAttribute('data-testid');
  const t = tr.querySelector('span.truncate[title]');
  const a = tr.querySelector('a[href]');
  return {
    key: tid,
    id: tid.replace('problem-', ''),
    title: t ? t.getAttribute('title') : '',
    cells: [...tr.querySelectorAll('td')].map(td => (td.textContent || '').replace(/\\s+/g, ' ').trim()),
    titled: [...tr.querySelectorAll('td span[title]')].map(s => s.getAttribute('title')),
    href: a ? a.getAttribute('href') : ''
  };
})
"""

JS_CARDS = """
() => {
  const hasH3 = d => !!d.querySelector('h3');
  const isCard = d => hasH3(d) && ![...d.querySelectorAll('div.group')].some(hasH3);
  const leaves = el => el
    ? [...el.querySelectorAll('*')].filter(e => e.children.length === 0)
        .map(e => (e.textContent || '').replace(/\\s+/g, ' ').trim()).filter(Boolean)
    : [];
  return [...document.querySelectorAll('div.group')].filter(isCard).map(c => {
    const h3 = c.querySelector('h3');
    const a = h3.closest('a') || c.querySelector('a[href]');
    const p = c.querySelector('p');
    const head = h3.previousElementSibling;
    const sibs = [...h3.parentElement.children];
    const after = sibs.slice(sibs.indexOf(h3) + 1).filter(e => e !== p);
    return {
      key: (h3.textContent || '').trim(),
      title: (h3.textContent || '').trim(),
      href: a ? a.getAttribute('href') : '',
      desc: p ? (p.textContent || '').replace(/\\s+/g, ' ').trim() : '',
      badges: leaves(head),
      meta: after.flatMap(leaves)
    };
  });
}
"""

JS_PAGINATION_HTML = """
() => {
  const b = [...document.querySelectorAll('button,a')].find(e => (e.textContent || '').trim() === '2');
  return b ? b.parentElement.outerHTML.slice(0, 1200) : '';
}
"""


# ------------------------------------------------------------------ helpers
def slugify(s):
    return re.sub(r"[^a-z0-9]+", "-", s.lower()).strip("-")


def pick_difficulty(texts):
    for t in texts:
        m = DIFF_RE.search(t or "")
        if m:
            return m.group(1).capitalize()
    return ""


def parse_table_row(r):
    try:
        cells = r.get("cells", [])
        title = r.get("title") or (max(cells, key=len) if cells else "")
        tags = [t for t in r.get("titled", []) if t and t != title]
        href = r.get("href", "")
        url = (BASE + href) if href.startswith("/") else f"{BASE}/problems/{r['id']}"
        return {
            "section": "problems", "id": r["id"], "title": title,
            "difficulty": pick_difficulty(cells),
            "category": tags[0] if tags else "", "tags": tags,
            "url": url, "url_source": "link" if href else "id",
            "access": "", "popularity": "", "steps": "",
            "description": "", "meta": [], "cells_raw": cells,
        }
    except Exception as e:  # noqa: BLE001
        print(f"    [!] bỏ qua 1 dòng lỗi: {e}")
        return None


def parse_card(section, r, state):
    try:
        title = r["title"]
        badges, meta = r.get("badges", []), r.get("meta", [])
        diff = pick_difficulty(badges) or pick_difficulty(meta)
        tags = [b for b in badges if not DIFF_RE.fullmatch(b) and not NUM_RE.match(b)
                and b.lower() not in ("premium", "free") and b != title]
        allb = [x.lower() for x in badges]
        access = "Premium" if "premium" in allb else "Free"  # thẻ miễn phí không có badge
        pop = next((b for b in badges if NUM_RE.match(b) and re.search(r"\d", b)), "")
        m = re.search(r"(\d+)\s*steps?", " ".join(meta), re.I)
        steps = m.group(1) if m else ""
        href = r.get("href", "")
        if href.startswith("/"):
            url, src = BASE + href, "link"
        else:
            prefix = state.get("real_prefix") or SECTIONS[section]["prefix"]
            url = f"{prefix}/{slugify(title)}"
            src = "slug-verified" if state.get("slug_ok") else "slug-guess"
        return {
            "section": section, "id": slugify(title), "title": title,
            "difficulty": diff, "category": tags[0] if tags else "", "tags": tags,
            "url": url, "url_source": src,
            "access": access, "popularity": pop, "steps": steps,
            "description": r.get("desc", ""), "meta": meta, "cells_raw": badges,
        }
    except Exception as e:  # noqa: BLE001
        print(f"    [!] bỏ qua 1 thẻ lỗi: {e}")
        return None


# ------------------------------------------------------------------ scraping
async def extract(page, mode):
    return await page.evaluate(JS_TABLE if mode == "table" else JS_CARDS)


async def probe_click(page, section, cfg, first_title, state):
    """Bấm thử bài đầu tiên để biết URL thật, rồi quay lại."""
    sel = 'tr[data-testid^="problem-"] span.truncate' if cfg["mode"] == "table" else "h3"
    try:
        await page.locator(sel).first.click(timeout=5000)
        await page.wait_for_timeout(2500)
        real = page.url
        print(f"    [probe] bấm '{first_title}' -> {real}")
        if cfg["mode"] == "cards":
            state["real_prefix"] = real.rsplit("/", 1)[0]
            state["slug_ok"] = real.rstrip("/").rsplit("/", 1)[-1] == slugify(first_title)
            print(f"    [probe] slug đoán {'KHỚP' if state['slug_ok'] else 'KHÔNG KHỚP'} -> "
                  f"{'URL các bài sẽ dựng từ tiêu đề' if state['slug_ok'] else 'URL chỉ là ước đoán!'}")
        await page.go_back(wait_until="domcontentloaded")
        await page.wait_for_selector("tr[data-testid^='problem-']" if cfg["mode"] == "table" else "h3",
                                     timeout=20000)
        await page.wait_for_timeout(1000)
    except Exception as e:  # noqa: BLE001
        print(f"    [probe] lỗi: {e}")
        if page.url.rstrip("/") != cfg["url"].rstrip("/"):
            await page.goto(cfg["url"], wait_until="domcontentloaded")
            await page.wait_for_timeout(2000)


NAV = 'nav[aria-label="Pagination"]'


async def total_pages(page):
    """Số trang lớn nhất đọc từ các nút aria-label='Page N' (vd 102)."""
    return await page.evaluate("""() => {
      const nav = document.querySelector('nav[aria-label="Pagination"]');
      if (!nav) return 0;
      const nums = [...nav.querySelectorAll('button[aria-label^="Page "]')]
        .map(b => parseInt(b.getAttribute('aria-label').replace('Page ', ''), 10))
        .filter(Number.isFinite);
      return nums.length ? Math.max(...nums) : 0;
    }""")


async def current_page(page):
    return await page.evaluate("""() => {
      const b = document.querySelector('nav[aria-label="Pagination"] button[aria-current="page"]');
      return b ? parseInt((b.textContent || '').trim(), 10) : 0;
    }""")


async def go_next(page, n, mode, before_keys):
    """Sang trang n+1: ưu tiên 'Next page', dự phòng nút 'Page n+1'."""
    nav = page.locator(NAV)
    options = [
        ("Next page", nav.locator('button[aria-label="Next page"]')),
        (f"Page {n + 1}", nav.locator(f'button[aria-label="Page {n + 1}"]')),
    ]
    for name, loc in options:
        try:
            if not await loc.count():
                continue
            el = loc.first
            if not await el.is_enabled():
                continue
            await el.scroll_into_view_if_needed(timeout=3000)
            await el.click(timeout=5000)
            for _ in range(60):  # tối đa ~18s chờ nội dung đổi
                await page.wait_for_timeout(300)
                keys = [r["key"] for r in await extract(page, mode)]
                if keys and keys != before_keys:
                    await page.wait_for_timeout(400)
                    return name
        except Exception:  # noqa: BLE001
            continue
    return None


async def scrape_section(browser, section, cfg):
    print(f"\n=== {section.upper()}  ->  {cfg['url']}")
    page = await browser.new_page(viewport={"width": 1440, "height": 900})
    store, state = {}, {}
    ready = "tr[data-testid^='problem-']" if cfg["mode"] == "table" else "h3"
    try:
        await page.goto(cfg["url"], wait_until="domcontentloaded", timeout=60000)
        await page.wait_for_selector(ready, timeout=30000)
        await page.wait_for_timeout(2500)

        raws = await extract(page, cfg["mode"])
        print(f"    trang 1 có {len(raws)} mục")
        total = await total_pages(page)
        print(f"    tổng số trang đọc từ thanh phân trang: {total or 'KHÔNG THẤY nav phân trang'}")

        needs_probe = not (raws and raws[0].get("href"))
        if raws and needs_probe:
            await probe_click(page, section, cfg, raws[0]["title"], state)

        last_n = 0
        for n in range(1, (total or MAX_PAGES) + 1):
            last_n = n
            raws = await extract(page, cfg["mode"])
            added = 0
            for r in raws:
                item = parse_table_row(r) if cfg["mode"] == "table" else parse_card(section, r, state)
                if item and item["url"] not in store:
                    store[item["url"]] = item
                    added += 1
            cur = await current_page(page)
            flag = "" if cur == n else f"  [!] thanh phân trang báo trang {cur}"
            print(f"    trang {n:>3}/{total or '?'}: {len(raws):>3} mục, +{added:>3} mới, tổng {len(store):>5}{flag}", flush=True)
            if n > 1 and added == 0:
                print("    -> trang không có bài mới, dừng.")
                break
            how = await go_next(page, n, cfg["mode"], [r["key"] for r in raws])
            if not how:
                print("    -> không còn trang kế, kết thúc.")
                break
            if n == 1:
                print(f"    (chuyển trang bằng: {how}; URL hiện tại: {page.url})")
        if total and last_n < total:
            print(f"    [!] CHỈ ĐI ĐƯỢC {last_n}/{total} TRANG - kết quả chưa đầy đủ")
        else:
            print(f"    OK: đã đi hết {last_n} trang")
    except Exception as e:  # noqa: BLE001
        print(f"    [!] lỗi ở mục {section}: {e}")
    finally:
        if len(store) < 5:
            with open(f"debug_{section}.html", "w", encoding="utf-8") as f:
                f.write(await page.content())
            await page.screenshot(path=f"debug_{section}.png")
            print(f"    [!] quá ít kết quả, đã lưu debug_{section}.html / .png")
        await page.close()
    return list(store.values())


async def main():
    stamp = time.strftime("%Y%m%d_%H%M%S")
    out = f"deepml_v2_{stamp}"
    results = []
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True, args=["--no-sandbox", "--disable-dev-shm-usage"])
        for name, cfg in SECTIONS.items():
            results.extend(await scrape_section(browser, name, cfg))
        await browser.close()

    with open(f"{out}.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)

    cols = ["section", "id", "title", "difficulty", "category", "tags", "url", "url_source",
            "access", "popularity", "steps", "description", "meta", "cells_raw"]
    with open(f"{out}.csv", "w", newline="", encoding="utf-8-sig") as f:
        w = csv.DictWriter(f, fieldnames=cols)
        w.writeheader()
        for r in results:
            row = dict(r)
            for k in ("tags", "meta", "cells_raw"):
                row[k] = " | ".join(r[k])
            w.writerow(row)

    print("\n" + "=" * 50)
    for name in SECTIONS:
        sub = [r for r in results if r["section"] == name]
        print(f"{name:<9}: {len(sub):>5} bài")
        for r in sub[:2]:
            print("   ", json.dumps(r, ensure_ascii=False)[:400])
    print(f"TỔNG     : {len(results)} bài  -> {out}.csv / .json")
    try:
        from google.colab import files  # type: ignore
        files.download(f"{out}.csv")
        files.download(f"{out}.json")
    except Exception:  # noqa: BLE001
        pass


def run():
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(main())
    import nest_asyncio
    nest_asyncio.apply()
    return loop.run_until_complete(main())


run()


=== PROBLEMS  ->  https://www.deep-ml.com/problems
    trang 1 có 15 mục
    tổng số trang đọc từ thanh phân trang: 102
    [probe] bấm 'Matrix-Vector Dot Product' -> https://www.deep-ml.com/problems/1
    trang   1/102:  15 mục, + 15 mới, tổng    15
    (chuyển trang bằng: Next page; URL hiện tại: https://www.deep-ml.com/problems?page=2)
    trang   2/102:  15 mục, + 15 mới, tổng    30
    trang   3/102:  15 mục, + 15 mới, tổng    45
    trang   4/102:  15 mục, + 15 mới, tổng    60
    trang   5/102:  15 mục, + 15 mới, tổng    75
    trang   6/102:  15 mục, + 15 mới, tổng    90
    trang   7/102:  15 mục, + 15 mới, tổng   105
    trang   8/102:  15 mục, + 15 mới, tổng   120
    trang   9/102:  15 mục, + 15 mới, tổng   135
    trang  10/102:  15 mục, + 15 mới, tổng   150
    trang  11/102:  15 mục, + 15 mới, tổng   165
    trang  12/102:  15 mục, + 15 mới, tổng   180
    trang  13/102:  15 mục, + 15 mới, tổng   195
    trang  14/102:  15 mục, + 15 mới, tổng   210
    trang  15/102:  1

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>